# 1. 教学智能体的设计

## 1.1. 业务任务与流程

### (1) 务流程说明
    1. 输入一个主题，教学要求
    2. 生成章（结构化输出）
    3. 生成节（结构化输出）
    4. 生成知识点（结构化输出）
    5. 质量检测（质量标准）
    6. 生成大纲（导出为excel/word，json）
    7. 输出excel文档

### (2) 业务流程图

```mermaid
flowchart LR
    A[输入一个主题，教学要求] --> B[生成章<br>结构化输出]
    B --> C[生成节<br>结构化输出]
    C --> D[生成知识点<br>结构化输出]
    D --> E[质量检测<br>质量标准]
    E -- 不通过 --> B
    E -- 通过 --> F[生成大纲<br>导出为 Excel/Word/JSON]
    F --> G[输出 Excel 文档]
```

## 1.2. 智能体模式的选择

### (1) 选择图模式实现自动化流程（交接模式，子代理）

- 因为业务是单纯的流程执行，不存在推理。
    - 任务结构：线性
        - 流程确定：
        - 状态管理：顺序传递（图模式）
    - 模式选择：使用langgraph实现自动化流程。

### (2) 与其他模式的对比分析

- ReAct：X不存在推理
- 图模式：最优选择，结构确定，状态顺序处理。
    - 单智能体
    - 多智能体（章内容生成使用Skill多智能体模式）
- 流水线模式：过于陈旧（函数调用的模式）。
    - 而是接口设计复杂，冗余。

### (3) 核心的设计原则

- 组件解耦：
    - 每个智能体的独立性：通过AgentState实现多个代理之间的数据结构。
- 流程透明：
    - 每个环节的数据状态都是客监控。
- 错误隔离：
    - 智能体之间不要传递错误。
- 顺序编排：
    - 使用LCEL能否实现。

## 1.3. 模式结构

```mermaid
graph TD
    subgraph "分层流水线架构 (Hierarchical Pipeline Pattern)"
        
        subgraph "控制层 - Orchestrator"
            CO[CourseOrchestrator<br/>课程大纲协调器]
            SM[StateManager<br/>状态管理器]
            QG[QualityGate<br/>质量门禁]
        end
        
        subgraph "执行层 - Pipeline Stages"
            direction LR
            
            subgraph "Stage1: 章生成层"
                CA[ChapterAgent<br/>章生成智能体]
                CV[ChapterValidator<br/>章质量校验]
            end
            
            subgraph "Stage2: 节生成层"
                SA[SectionAgent<br/>节生成智能体]
                SV[SectionValidator<br/>节质量校验]
            end
            
            subgraph "Stage3: 知识点生成层"
                KPA[KnowledgePointAgent<br/>知识点生成智能体]
                KPV[KnowledgePointValidator<br/>知识点质量校验]
            end
        end
        
        subgraph "基础能力层"
            LLM[LLM Provider<br/>GPT/DeepSeek/Claude]
            PROMPT[Prompt Template<br/>提示词模板库]
            SCHEMA[Output Schema<br/>结构化输出定义]
        end
    end
    
    CO --> CA
    CO --> SA
    CO --> KPA
    
    CA --> CV
    CV -->|passed| SM
    CV -->|failed| CA
    
    SA --> SV
    SV -->|passed| SM
    SV -->|failed| SA
    
    KPA --> KPV
    KPV -->|passed| SM
    KPV -->|failed| KPA
    
    CA -.-> LLM
    SA -.-> LLM
    KPA -.-> LLM
    
    SM --> SCHEMA
    SCHEMA --> CO
```

### (1) 核心Agent组件

| 组件 | 职责 | 关键特性 |
|------|------|---------|
| **CourseOrchestrator** | 控制整体生成流程，协调各智能体工作 | 流程编排、异常处理、进度追踪 |
| **ChapterAgent** | 根据课程主题生成章节目录 | 结构化输出、层级合理性校验 |
| **SectionAgent** | 为每个章节生成下属节 | 依赖章节上下文、数量控制(2-5节) |
| **KnowledgePointAgent** | 为每个节生成知识点 | 粒度控制、难度分级、时长预估 |
| **QualityGate** | 每一层输出质量检查 | 完整性、一致性、合理性校验 |
| **StateManager** | 管理跨层状态传递 | 上下文缓存、断点续传 |

### (2) 状态AgentState设计

```mermaid
flowchart LR
    subgraph Input
        TOPIC[课程主题]
        LEVEL[难度级别]
        AUDIENCE[目标受众]
    end
    
    subgraph Stage1
        CH1[第1章] --> CH2[第2章] --> CH3[第N章]
    end
    
    subgraph Stage2
        CH1 --> S11[1.1节] --> S12[1.2节]
        CH2 --> S21[2.1节] --> S22[2.2节]
    end
    
    subgraph Stage3
        S11 --> KP111[知识点1.1.1] --> KP112[知识点1.1.2]
        S12 --> KP121[知识点1.2.1]
    end
    
    TOPIC --> Stage1
    LEVEL --> Stage1
    AUDIENCE --> Stage1
    
    Stage1 --> Stage2
    Stage2 --> Stage3
    
    Stage3 --> OUTPUT[完整课程大纲JSON]
```

## 1.4. 自动化流程图

```mermaid
flowchart TD
    START([开始]) --> INIT[初始化LLM和智能体]
    INIT --> INPUT[接收课程主题/难度/受众]
    
    INPUT --> PHASE1[阶段1: 生成章节]
    
    subgraph PHASE1_DETAIL [阶段1详情]
        P1_GEN[ChapterAgent.generate_chapters] --> P1_VALID{质量检查}
        P1_VALID -->|通过| P1_SAVE[保存章节列表]
        P1_VALID -->|不通过且重试<3| P1_GEN
        P1_VALID -->|不通过且重试=3| P1_FALLBACK[使用默认章节结构]
        P1_FALLBACK --> P1_SAVE
    end

    PHASE1 --> PHASE1_DETAIL
    
    P1_SAVE --> PHASE2[阶段2: 遍历章节生成节]
    
    subgraph PHASE2_DETAIL [阶段2详情]
        P2_LOOP{还有章节？} -->|是| P2_NEXT[取下一章节]
        P2_NEXT --> P2_GEN[SectionAgent.generate_sections]
        P2_GEN --> P2_VALID{质量检查}
        P2_VALID -->|通过| P2_ATTACH[节附加到章节]
        P2_VALID -->|不通过且重试<3| P2_GEN
        P2_VALID -->|不通过且重试=3| P2_DEFAULT[使用默认节结构]
        P2_DEFAULT --> P2_ATTACH
        P2_ATTACH --> P2_LOOP
        P2_LOOP -->|否| P2_DONE[完成所有章节的节生成]
    end
    PHASE2 --> PHASE2_DETAIL
    P2_DONE --> PHASE3[阶段3: 遍历节生成知识点]
    
    subgraph PHASE3_DETAIL [阶段3详情]
        P3_LOOP{还有节？} -->|是| P3_NEXT[取下一节]
        P3_NEXT --> P3_GEN[KnowledgePointAgent.generate_points]
        P3_GEN --> P3_VALID{质量检查}
        P3_VALID -->|通过| P3_ATTACH[知识点附加到节]
        P3_VALID -->|不通过且重试<3| P3_GEN
        P3_VALID -->|不通过且重试=3| P3_DEFAULT[使用默认知识点]
        P3_DEFAULT --> P3_ATTACH
        P3_ATTACH --> P3_LOOP
        P3_LOOP -->|否| P3_DONE
    end
    PHASE3 --> PHASE3_DETAIL
    
    
    P3_DONE --> OUTPUT[组装完整大纲]
    OUTPUT --> SAVE[保存JSON文件]
    SAVE --> DISPLAY[打印大纲到控制台]
    DISPLAY --> END([结束])
```

# 2. 智能体的代码测试

## 2.1. 状态定义

In [ ]:
# ```python
class DifficultyLevel(str, Enum):
    """难度级别枚举"""
    BEGINNER = "beginner"
    INTERMEDIATE = "intermediate"
    ADVANCED = "advanced"

class QualityLevel(str, Enum):
    """质量等级枚举"""
    EXCELLENT = "excellent"
    GOOD = "good"
    ACCEPTABLE = "acceptable"
    NEEDS_IMPROVEMENT = "needs_improvement"
    REJECTED = "rejected"

class KnowledgePoint(BaseModel):
    """知识点模型 - LangChain 1.2结构化输出"""
    name: str = Field(description="知识点名称")
    description: str = Field(description="知识点详细描述")
    difficulty_level: str = Field(default="medium", description="难度级别: easy/medium/hard")
    estimated_time_minutes: int = Field(default=30, description="预估学习时间(分钟)")
    prerequisites: List[str] = Field(default_factory=list, description="前置知识点列表")

class Section(BaseModel):
    """节模型"""
    name: str = Field(description="节名称")
    description: str = Field(description="节详细描述")
    learning_objectives: List[str] = Field(default_factory=list, description="学习目标列表")
    knowledge_points: List[KnowledgePoint] = Field(default_factory=list, description="知识点列表")

class Chapter(BaseModel):
    """章模型"""
    name: str = Field(description="章节名称")
    description: str = Field(description="章节描述")
    learning_objectives: List[str] = Field(default_factory=list, description="学习目标列表")
    sections: List[Section] = Field(default_factory=list, description="节列表")

class CourseOutline(BaseModel):
    """课程大纲完整模型"""
    course_name: str = Field(description="课程名称")
    course_description: str = Field(description="课程描述")
    total_duration_hours: int = Field(default=40, description="总学时(小时)")
    difficulty_level: str = Field(default="intermediate", description="难度级别")
    target_audience: str = Field(default="general", description="目标受众")
    prerequisites: List[str] = Field(default_factory=list, description="前置知识要求")
    chapters: List[Chapter] = Field(default_factory=list, description="章节列表")
    created_at: str = Field(default_factory=lambda: datetime.now().isoformat(), description="创建时间")

class QualityReport(BaseModel):
    """质量检查报告"""
    quality_level: QualityLevel
    score: float = Field(ge=0, le=100)
    issues: List[str] = Field(default_factory=list)
    suggestions: List[str] = Field(default_factory=list)
    passed: bool
# ```

## 2.2. 系统提示词

In [ ]:
# ==================== Prompt模板库 ====================

CHAPTER_GENERATION_PROMPT = """你是一位资深的课程设计专家，拥有10年以上教学大纲设计经验。

## 课程信息
- 课程名称: {course_name}
- 课程描述: {course_description}
- 难度级别: {difficulty_level}
- 目标受众: {target_audience}

## 任务要求
请为上述课程设计完整的章节目录结构。

## 设计原则
1. 章节数量: 3-8章（根据课程深度合理分配）
2. 逻辑递进: 从基础概念→核心知识→高级应用→实战项目
3. 每章应有明确的学习目标（3-5个）
4. 章节名称需简洁明了，体现核心内容

## 输出格式
请严格按照以下JSON格式输出，不要包含任何其他内容：

```json
{{
    "chapters": [
        {{
            "name": "章节名称",
            "description": "章节详细描述（50-100字）",
            "learning_objectives": ["学习目标1", "学习目标2", "学习目标3"]
        }}
    ]
}}
```

请开始设计："""

SECTION_GENERATION_PROMPT = """你是一位课程内容细化专家，擅长将章节拆解为具体的节。

## 上下文信息
- 课程名称: {course_name}
- 章节名称: {chapter_name}
- 章节描述: {chapter_description}
- 章节学习目标: {chapter_objectives}

## 任务要求
请将上述章节细分为2-5个节（sub-sections）。

## 设计原则
1. 每节应有明确的学习重点
2. 节与节之间要有逻辑递进关系
3. 每节学习目标要具体、可衡量
4. 节时长建议: 2-4小时/节

## 输出格式
```json
{{
    "sections": [
        {{
            "name": "节名称",
            "description": "节详细描述",
            "learning_objectives": ["学习目标1", "学习目标2"]
        }}
    ]
}}
```

请开始设计："""

KNOWLEDGE_POINT_PROMPT = """你是一位教育内容专家，擅长将节拆解为最小的教学单元——知识点。

## 上下文信息
- 课程名称: {course_name}
- 所属章节: {chapter_name}
- 节名称: {section_name}
- 节描述: {section_description}
- 学习目标: {section_objectives}

## 任务要求
请将上述节细化为3-6个知识点。

## 设计原则
1. 每个知识点应是独立、完整的最小教学单元
2. 难度递进: easy → medium → hard
3. 每个知识点预估学习时间: 15-45分钟
4. 标注知识点之间的前置依赖关系

## 输出格式
```json
{{
    "knowledge_points": [
        {{
            "name": "知识点名称",
            "description": "知识点详细说明",
            "difficulty_level": "easy/medium/hard",
            "estimated_time_minutes": 30,
            "prerequisites": ["前置知识点名称"]
        }}
    ]
}}
```

请开始设计："""

QUALITY_CHECK_PROMPT = """你是一位课程质量评估专家，请评估以下{content_type}的质量。

## 评估内容
{content}

## 评估维度
1. 完整度 (0-30分): 是否完整覆盖了所需信息
2. 逻辑性 (0-30分): 结构是否合理，是否有清晰的逻辑递进
3. 可行性 (0-20分): 是否适合实际教学场景
4. 专业性 (0-20分): 术语使用是否准确，内容是否专业

## 输出格式
```json
{{
    "quality_level": "excellent/good/acceptable/needs_improvement/rejected",
    "score": 总分(0-100),
    "issues": ["问题1", "问题2"],
    "suggestions": ["改进建议1"],
    "passed": true/false
}}
```

通过标准: 总分 >= 70分

请开始评估："""


## 2.3. 智能体实现

In [ ]:
class BaseAgent:
    """智能体基类 - 封装LLM调用和重试逻辑"""
    
    def __init__(self, llm, max_retries: int = 3, retry_delay: int = 2):
        self.llm = llm
        self.max_retries = max_retries
        self.retry_delay = retry_delay
        self.output_parser = StrOutputParser()
    
    def _call_llm_with_retry(self, prompt_template: ChatPromptTemplate, context: Dict[str, Any]) -> str:
        """带重试和指数退避的LLM调用"""
        for attempt in range(self.max_retries):
            try:
                # 使用LangChain 1.2的invoke方法
                chain = prompt_template | self.llm | self.output_parser
                response = chain.invoke(context)
                return response
            except Exception as e:
                print(f"    ⚠️ 第{attempt+1}次LLM调用失败: {e}")
                if attempt < self.max_retries - 1:
                    time.sleep(self.retry_delay * (attempt + 1))
                else:
                    raise
        return ""
    
    def _extract_json(self, response: str) -> Dict:
        """从LLM响应中提取JSON"""
        response = response.strip()
        
        # 处理markdown代码块
        if "```json" in response:
            start = response.find("```json") + 7
            end = response.find("```", start)
            response = response[start:end].strip()
        elif "```" in response:
            start = response.find("```") + 3
            end = response.find("```", start)
            response = response[start:end].strip()
        
        # 尝试直接解析
        try:
            return json.loads(response)
        except json.JSONDecodeError:
            # 尝试找到第一个{和最后一个}
            start_idx = response.find("{")
            end_idx = response.rfind("}") + 1
            if start_idx != -1 and end_idx != 0:
                return json.loads(response[start_idx:end_idx])
            raise
    
    def get_system_prompt(self) -> str:
        """获取系统提示词 - 子类可重写"""
        return "你是一个专业的AI助手。"

# ==================== 专门智能体实现 ====================

class ChapterAgent(BaseAgent):
    """章生成智能体 - 负责生成课程章节"""
    
    def generate_chapters(self, course_name: str, course_description: str, 
                          difficulty_level: str, target_audience: str) -> List[Dict]:
        """生成课程章节"""
        prompt = ChatPromptTemplate.from_messages([
            ("system", "你是一位资深的课程设计专家。请严格按照JSON格式输出。"),
            ("human", CHAPTER_GENERATION_PROMPT)
        ])
        
        context = {
            "course_name": course_name,
            "course_description": course_description,
            "difficulty_level": difficulty_level,
            "target_audience": target_audience
        }
        
        try:
            response = self._call_llm_with_retry(prompt, context)
            data = self._extract_json(response)
            chapters = data.get("chapters", [])
            print(f"    ✅ 成功生成 {len(chapters)} 个章节")
            return chapters
        except Exception as e:
            print(f"    ❌ 章节生成失败: {e}")
            return self._get_fallback_chapters(course_name)
    
    def _get_fallback_chapters(self, course_name: str) -> List[Dict]:
        """降级方案：返回默认章节结构"""
        return [
            {
                "name": f"{course_name} - 基础入门",
                "description": "掌握课程的基础知识和核心概念",
                "learning_objectives": ["理解基本概念", "掌握核心术语", "能够完成基础操作"]
            },
            {
                "name": f"{course_name} - 核心进阶",
                "description": "深入学习课程的核心内容和实战技巧",
                "learning_objectives": ["掌握核心技能", "独立完成实战任务", "解决常见问题"]
            },
            {
                "name": f"{course_name} - 高级实战",
                "description": "学习高级技巧并通过项目实战巩固",
                "learning_objectives": ["掌握高级技术", "完成综合项目", "具备独立开发能力"]
            }
        ]

class SectionAgent(BaseAgent):
    """节生成智能体 - 负责为章节生成节"""
    
    def generate_sections(self, chapter: Dict, course_name: str) -> List[Dict]:
        """为章节生成节"""
        prompt = ChatPromptTemplate.from_messages([
            ("system", "你是一位课程内容细化专家。请严格按照JSON格式输出。"),
            ("human", SECTION_GENERATION_PROMPT)
        ])
        
        context = {
            "course_name": course_name,
            "chapter_name": chapter.get("name", ""),
            "chapter_description": chapter.get("description", ""),
            "chapter_objectives": ", ".join(chapter.get("learning_objectives", []))
        }
        
        try:
            response = self._call_llm_with_retry(prompt, context)
            data = self._extract_json(response)
            sections = data.get("sections", [])
            return sections
        except Exception as e:
            print(f"    ⚠️ 节生成失败，使用默认结构: {e}")
            return self._get_fallback_sections(chapter)
    
    def _get_fallback_sections(self, chapter: Dict) -> List[Dict]:
        """降级方案：默认节结构"""
        chapter_name = chapter.get("name", "本章")
        return [
            {
                "name": f"{chapter_name} - 核心概念",
                "description": f"理解{chapter_name}的核心概念和基本原理",
                "learning_objectives": ["理解核心概念", "掌握基础知识"]
            },
            {
                "name": f"{chapter_name} - 实践应用",
                "description": f"掌握{chapter_name}的实际应用技能",
                "learning_objectives": ["能够独立实践", "解决实际问题"]
            }
        ]

class KnowledgePointAgent(BaseAgent):
    """知识点生成智能体 - 负责为节生成知识点"""
    
    def generate_knowledge_points(self, section: Dict, chapter: Dict, course_name: str) -> List[Dict]:
        """为节生成知识点"""
        prompt = ChatPromptTemplate.from_messages([
            ("system", "你是一位教育内容专家。请严格按照JSON格式输出。"),
            ("human", KNOWLEDGE_POINT_PROMPT)
        ])
        
        context = {
            "course_name": course_name,
            "chapter_name": chapter.get("name", ""),
            "section_name": section.get("name", ""),
            "section_description": section.get("description", ""),
            "section_objectives": ", ".join(section.get("learning_objectives", []))
        }
        
        try:
            response = self._call_llm_with_retry(prompt, context)
            data = self._extract_json(response)
            knowledge_points = data.get("knowledge_points", [])
            return knowledge_points
        except Exception as e:
            print(f"      ⚠️ 知识点生成失败，使用默认结构: {e}")
            return self._get_fallback_knowledge_points(section)
    
    def _get_fallback_knowledge_points(self, section: Dict) -> List[Dict]:
        """降级方案：默认知识点结构"""
        section_name = section.get("name", "本节")
        return [
            {
                "name": f"{section_name} - 概念理解",
                "description": f"理解{section_name}的核心概念",
                "difficulty_level": "easy",
                "estimated_time_minutes": 20,
                "prerequisites": []
            },
            {
                "name": f"{section_name} - 实践操作",
                "description": f"掌握{section_name}的实践技能",
                "difficulty_level": "medium",
                "estimated_time_minutes": 30,
                "prerequisites": [f"{section_name} - 概念理解"]
            },
            {
                "name": f"{section_name} - 综合应用",
                "description": f"综合运用{section_name}解决实际问题",
                "difficulty_level": "hard",
                "estimated_time_minutes": 40,
                "prerequisites": [f"{section_name} - 实践操作"]
            }
        ]

class QualityAgent(BaseAgent):
    """质量检查智能体 - 负责评估生成内容质量"""
    
    def check_quality(self, content: Dict, content_type: str) -> QualityReport:
        """检查内容质量"""
        prompt = ChatPromptTemplate.from_messages([
            ("system", "你是一位课程质量评估专家。请严格按照JSON格式输出。"),
            ("human", QUALITY_CHECK_PROMPT)
        ])
        
        context = {
            "content_type": content_type,
            "content": json.dumps(content, ensure_ascii=False, indent=2)
        }
        
        try:
            response = self._call_llm_with_retry(prompt, context)
            data = self._extract_json(response)
            return QualityReport(
                quality_level=QualityLevel(data.get("quality_level", "acceptable")),
                score=data.get("score", 70),
                issues=data.get("issues", []),
                suggestions=data.get("suggestions", []),
                passed=data.get("passed", True)
            )
        except Exception as e:
            print(f"    ⚠️ 质量检查失败，默认通过: {e}")
            return QualityReport(
                quality_level=QualityLevel.ACCEPTABLE,
                score=70,
                issues=[],
                suggestions=[],
                passed=True
            )


## 2.4. 工程模式实现（自动化流程实现）

In [ ]:
class StateManager:
    """状态管理器 - 管理生成进度和上下文传递"""
    
    def __init__(self):
        self._state: Dict[str, Any] = {}
        self._history: List[Dict] = []
        self._checkpoints: Dict[str, Dict] = {}
    
    def set(self, key: str, value: Any) -> None:
        """设置状态"""
        self._state[key] = value
        self._history.append({
            "timestamp": datetime.now().isoformat(),
            "action": f"set_{key}",
            "value": str(value)[:200]
        })
    
    def get(self, key: str, default=None) -> Any:
        """获取状态"""
        return self._state.get(key, default)
    
    def checkpoint(self, name: str) -> None:
        """保存检查点"""
        self._checkpoints[name] = {
            "state": self._state.copy(),
            "timestamp": datetime.now().isoformat()
        }
        print(f"    📍 已保存检查点: {name}")
    
    def restore(self, name: str) -> bool:
        """恢复检查点"""
        if name in self._checkpoints:
            self._state = self._checkpoints[name]["state"].copy()
            print(f"    🔄 已恢复检查点: {name}")
            return True
        return False
    
    def get_summary(self) -> Dict:
        """获取状态摘要"""
        return {
            "chapters_count": len(self.get("chapters", [])),
            "has_full_chapters": self.get("full_chapters") is not None,
            "has_complete_outline": self.get("complete_outline") is not None,
            "checkpoints": list(self._checkpoints.keys())
        }

# ==================== 课程大纲协调器 (主控制器) ====================

class CourseOutlineOrchestrator:
    """
    课程大纲协调器 - 分层流水线模式主控制器
    
    职责:
    1. 协调各智能体的工作流程
    2. 管理生成进度和状态
    3. 处理异常和降级
    """
    
    def __init__(self, llm):
        self.llm = llm
        self.state_manager = StateManager()
        
        # 初始化智能体
        self.chapter_agent = ChapterAgent(llm)
        self.section_agent = SectionAgent(llm)
        self.knowledge_point_agent = KnowledgePointAgent(llm)
        self.quality_agent = QualityAgent(llm)
        
        # 配置参数
        self.max_retries_per_stage = 3
    
    def generate(self, course_name: str, course_description: str,
                 difficulty_level: str = "intermediate",
                 target_audience: str = "general",
                 total_hours: int = 40) -> CourseOutline:
        """
        生成完整课程大纲 - 主入口方法
        """
        print("\n" + "="*70)
        print(f"📚 课程大纲生成器启动")
        print(f"   课程: {course_name}")
        print(f"   难度: {difficulty_level}")
        print(f"   受众: {target_audience}")
        print("="*70)
        
        # 保存课程基本信息
        course_info = {
            "course_name": course_name,
            "course_description": course_description,
            "difficulty_level": difficulty_level,
            "target_audience": target_audience,
            "total_hours": total_hours
        }
        self.state_manager.set("course_info", course_info)
        
        # 阶段1: 生成章节
        print("\n📖 阶段1: 生成章节结构")
        chapters = self._stage_generate_chapters(course_info)
        if not chapters:
            raise RuntimeError("章节生成失败，无法继续")
        self.state_manager.set("chapters", chapters)
        self.state_manager.checkpoint("after_chapters")
        
        # 阶段2: 生成节
        print("\n📌 阶段2: 生成节结构")
        full_chapters = self._stage_generate_sections(chapters, course_name)
        self.state_manager.set("full_chapters", full_chapters)
        self.state_manager.checkpoint("after_sections")
        
        # 阶段3: 生成知识点
        print("\n🎯 阶段3: 生成知识点")
        complete_chapters = self._stage_generate_knowledge_points(full_chapters, course_name)
        self.state_manager.set("complete_chapters", complete_chapters)
        
        # 阶段4: 组装最终输出
        print("\n📦 阶段4: 组装最终大纲")
        outline = self._assemble_outline(course_info, complete_chapters)
        self.state_manager.set("complete_outline", outline.model_dump())
        
        # 打印统计信息
        self._print_statistics(outline)
        
        print("\n" + "="*70)
        print("✅ 课程大纲生成完成!")
        print("="*70)
        
        return outline
    
    def _stage_generate_chapters(self, course_info: Dict) -> List[Dict]:
        """阶段1: 生成章节 (带质量检查和重试)"""
        for attempt in range(self.max_retries_per_stage):
            print(f"  🚀 第{attempt+1}次尝试生成章节...")
            
            chapters = self.chapter_agent.generate_chapters(
                course_info["course_name"],
                course_info["course_description"],
                course_info["difficulty_level"],
                course_info["target_audience"]
            )
            
            if not chapters:
                print(f"    ❌ 未生成任何章节")
                continue
            
            # 质量检查
            quality = self.quality_agent.check_quality(
                {"chapters": chapters, "count": len(chapters)},
                "课程章节"
            )
            print(f"    📊 质量评分: {quality.score}/100 - {quality.quality_level.value}")
            
            if quality.passed:
                print(f"    ✅ 章节质量合格，通过")
                return chapters
            else:
                print(f"    ⚠️ 章节质量不合格: {quality.issues}")
                if quality.suggestions:
                    print(f"    💡 改进建议: {quality.suggestions}")
        
        print(f"  ⚠️ 达到最大重试次数，使用降级方案")
        return self.chapter_agent._get_fallback_chapters(course_info["course_name"])
    
    def _stage_generate_sections(self, chapters: List[Dict], course_name: str) -> List[Dict]:
        """阶段2: 为所有章节生成节"""
        full_chapters = []
        
        for idx, chapter in enumerate(chapters, 1):
            print(f"\n  📕 处理第{idx}章: {chapter.get('name', '未命名')}")
            
            sections = self.section_agent.generate_sections(chapter, course_name)
            
            if not sections:
                print(f"    ⚠️ 未生成节，使用降级方案")
                sections = self.section_agent._get_fallback_sections(chapter)
            
            print(f"    ✅ 生成了 {len(sections)} 个节")
            
            # 为每个节打印简要信息
            for s_idx, sec in enumerate(sections, 1):
                print(f"      {idx}.{s_idx} {sec.get('name', '')[:40]}")
            
            chapter_with_sections = chapter.copy()
            chapter_with_sections["sections"] = sections
            full_chapters.append(chapter_with_sections)
        
        return full_chapters
    
    def _stage_generate_knowledge_points(self, chapters: List[Dict], course_name: str) -> List[Dict]:
        """阶段3: 为所有节生成知识点"""
        complete_chapters = []
        total_kps = 0
        
        for ch_idx, chapter in enumerate(chapters, 1):
            print(f"\n  📗 处理第{ch_idx}章: {chapter.get('name', '未命名')[:30]}")
            updated_sections = []
            
            sections = chapter.get("sections", [])
            for sec_idx, section in enumerate(sections, 1):
                print(f"    📍 处理节 {ch_idx}.{sec_idx}: {section.get('name', '')[:35]}")
                
                knowledge_points = self.knowledge_point_agent.generate_knowledge_points(
                    section, chapter, course_name
                )
                
                if not knowledge_points:
                    knowledge_points = self.knowledge_point_agent._get_fallback_knowledge_points(section)
                
                print(f"      ✅ 生成了 {len(knowledge_points)} 个知识点")
                total_kps += len(knowledge_points)
                
                # 打印知识点概要
                for kp in knowledge_points[:2]:
                    print(f"        • {kp.get('name', '')[:35]} [{kp.get('difficulty_level', 'medium')}]")
                if len(knowledge_points) > 2:
                    print(f"        ... 还有 {len(knowledge_points)-2} 个知识点")
                
                section_with_points = section.copy()
                section_with_points["knowledge_points"] = knowledge_points
                updated_sections.append(section_with_points)
            
            complete_chapter = chapter.copy()
            complete_chapter["sections"] = updated_sections
            complete_chapters.append(complete_chapter)
        
        print(f"\n  📊 阶段3完成: 共生成 {total_kps} 个知识点")
        return complete_chapters
    
    def _assemble_outline(self, course_info: Dict, complete_chapters: List[Dict]) -> CourseOutline:
        """阶段4: 组装最终大纲"""
        chapters = []
        
        for ch_dict in complete_chapters:
            sections = []
            for sec_dict in ch_dict.get("sections", []):
                knowledge_points = [
                    KnowledgePoint(
                        name=kp.get("name", ""),
                        description=kp.get("description", ""),
                        difficulty_level=kp.get("difficulty_level", "medium"),
                        estimated_time_minutes=kp.get("estimated_time_minutes", 30),
                        prerequisites=kp.get("prerequisites", [])
                    )
                    for kp in sec_dict.get("knowledge_points", [])
                ]
                
                section = Section(
                    name=sec_dict.get("name", ""),
                    description=sec_dict.get("description", ""),
                    learning_objectives=sec_dict.get("learning_objectives", []),
                    knowledge_points=knowledge_points
                )
                sections.append(section)
            
            chapter = Chapter(
                name=ch_dict.get("name", ""),
                description=ch_dict.get("description", ""),
                learning_objectives=ch_dict.get("learning_objectives", []),
                sections=sections
            )
            chapters.append(chapter)
        
        return CourseOutline(
            course_name=course_info["course_name"],
            course_description=course_info["course_description"],
            total_duration_hours=course_info["total_hours"],
            difficulty_level=course_info["difficulty_level"],
            target_audience=course_info["target_audience"],
            prerequisites=[],
            chapters=chapters
        )
    
    def _print_statistics(self, outline: CourseOutline) -> None:
        """打印统计信息"""
        total_sections = sum(len(ch.sections) for ch in outline.chapters)
        print([kp for ch in outline.chapters  for s in ch.sections  for kp in s.knowledge_points])
        total_kps = 0
        # total_kps = sum(
        #     len(kp) for ch in outline.chapters 
        #     for s in ch.sections 
        #     for kp in s.knowledge_points
        # )
        
        print("\n" + "-"*50)
        print("📊 生成统计")
        print("-"*50)
        print(f"  章节数: {len(outline.chapters)}")
        print(f"  节数: {total_sections}")
        print(f"  知识点数: {total_kps}")
        
        # 按难度统计知识点
        difficulty_count = {"easy": 0, "medium": 0, "hard": 0}
        for ch in outline.chapters:
            for s in ch.sections:
                for kp in s.knowledge_points:
                    difficulty_count[kp.difficulty_level] = difficulty_count.get(kp.difficulty_level, 0) + 1
        
        print(f"\n  知识点难度分布:")
        print(f"    简单: {difficulty_count['easy']} 个")
        print(f"    中等: {difficulty_count['medium']} 个")
        print(f"    困难: {difficulty_count['hard']} 个")
    
    def save_to_json(self, outline: CourseOutline, filename: str = "course_outline.json") -> None:
        """保存大纲到JSON文件"""
        outline_dict = outline.dict()
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(outline_dict, f, ensure_ascii=False, indent=2)
        print(f"\n💾 大纲已保存至: {filename}")
    
    def print_outline(self, outline: CourseOutline, max_kp_per_section: int = 3) -> None:
        """打印大纲到控制台"""
        print("\n" + "="*80)
        print(f"📚 课程大纲: {outline.course_name}")
        print("="*80)
        print(f"📝 描述: {outline.course_description}")
        print(f"⭐ 难度: {outline.difficulty_level}")
        print(f"👥 目标受众: {outline.target_audience}")
        print(f"⏱️  总学时: {outline.total_duration_hours}小时")
        print(f"📅 创建时间: {outline.created_at}")
        
        for ch_idx, chapter in enumerate(outline.chapters, 1):
            print(f"\n{'='*60}")
            print(f"第{ch_idx}章: {chapter.name}")
            print(f"{'='*60}")
            print(f"  📖 {chapter.description}")
            print(f"\n  🎯 本章学习目标:")
            for obj in chapter.learning_objectives:
                print(f"     • {obj}")
            
            for sec_idx, section in enumerate(chapter.sections, 1):
                print(f"\n  ┌─ {ch_idx}.{sec_idx} {section.name}")
                print(f"  │  📖 {section.description}")
                
                if section.learning_objectives:
                    print(f"  │  🎯 学习目标:")
                    for obj in section.learning_objectives[:2]:
                        print(f"  │     • {obj}")
                
                print(f"  │  📌 知识点:")
                for kp_idx, kp in enumerate(section.knowledge_points[:max_kp_per_section], 1):
                    time_str = f"{kp.estimated_time_minutes}分钟"
                    print(f"  │     {kp_idx}. {kp.name}")
                    print(f"  │        [{kp.difficulty_level}] {time_str}")
                    print(f"  │        {kp.description[:80]}...")
                
                if len(section.knowledge_points) > max_kp_per_section:
                    print(f"  │     ... 还有 {len(section.knowledge_points)-max_kp_per_section} 个知识点")
        
        print("\n" + "="*80)

## 2.5. 业务模块实现 

## 2.6. 前端实现

## 2.7. 控制器实现

- 调用业务模块，实现jinja模版，实现数据渲染。